In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
InputNo = 10
# 데이터 생성 함수
def generate_data(num_samples):
    X = np.random.randint(0, 10, (num_samples, InputNo))
    y = np.flip(X, axis=1)
    return X, y

# 데이터 생성
num_samples = 20000
X, y = generate_data(num_samples)

# 데이터 형태 변환 (LSTM을 사용하기 위해 3D 텐서로 변환)
X = X.reshape((X.shape[0], X.shape[1], 1))
y = y.reshape((y.shape[0], y.shape[1], 1))

# 인코더 정의
encoder_inputs = Input(shape=(InputNo, 1))
encoder = LSTM(128, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# 디코더 정의
decoder_inputs = Input(shape=(InputNo, 1))
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(10, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# 모델 정의
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 입력 데이터와 타겟 데이터를 동일하게 맞춤
decoder_input_data = np.zeros_like(X)

# EarlyStopping 콜백 정의
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 학습
model.fit([X, decoder_input_data], y, epochs=50, validation_split=0.2, callbacks=[early_stopping])

# 모델 검증 함수
def predict_reverse(input_sequence):
    input_sequence = np.array(input_sequence).reshape((1, InputNo, 1))
    decoder_input = np.zeros((1, InputNo, 1))
    predicted_sequence = model.predict([input_sequence, decoder_input])
    return np.argmax(predicted_sequence, axis=-1).reshape((InputNo,))
'''
# 입력 반복 루프
while True:
    user_input = input("입력 (쉼표로 구분된 5개의 숫자, 종료하려면 'quit' 입력): ")
    if user_input.lower() == "quit":
        break
    try:
        input_sequence = list(map(int, user_input.split(',')))
        if len(input_sequence) != 5:
            print("5개의 숫자를 입력해 주세요.")
            continue
        predicted_output = predict_reverse(input_sequence)
        print(f"입력: {input_sequence}")
        print(f"예측된 출력: {predicted_output}")
    except ValueError:
        print("유효한 숫자를 입력해 주세요.")
'''

Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.1735 - loss: 2.1415 - val_accuracy: 0.4219 - val_loss: 1.4804
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.5209 - loss: 1.2795 - val_accuracy: 0.7557 - val_loss: 0.8204
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8149 - loss: 0.7216 - val_accuracy: 0.9081 - val_loss: 0.5005
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9191 - loss: 0.4470 - val_accuracy: 0.9572 - val_loss: 0.3014
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9547 - loss: 0.2744 - val_accuracy: 0.9609 - val_loss: 0.2048
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9783 - loss: 0.1582 - val_accuracy: 0.9782 - val_loss: 0.1247
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9844 - loss: 0.1038 - val_accuracy: 0.9872 - val_loss: 0.0823
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9902 - loss: 0.0686 - val_accuracy: 0.

'\n# 입력 반복 루프\nwhile True:\n    user_input = input("입력 (쉼표로 구분된 5개의 숫자, 종료하려면 \'quit\' 입력): ")\n    if user_input.lower() == "quit":\n        break\n    try:\n        input_sequence = list(map(int, user_input.split(\',\')))\n        if len(input_sequence) != 5:\n            print("5개의 숫자를 입력해 주세요.")\n            continue\n        predicted_output = predict_reverse(input_sequence)\n        print(f"입력: {input_sequence}")\n        print(f"예측된 출력: {predicted_output}")\n    except ValueError:\n        print("유효한 숫자를 입력해 주세요.")\n'

In [3]:
# 검증용 데이터 생성
test_X, test_y = generate_data(20)

# 모델 예측 및 정확도 계산
correct_predictions = 0
total_predictions = len(test_X)

for i in range(total_predictions):
    input_sequence = test_X[i]
    true_output = test_y[i].reshape(InputNo,)
    predicted_output = predict_reverse(input_sequence)
    
    if np.array_equal(predicted_output, true_output):
        correct_predictions += 1

    print(f"입력: {input_sequence.reshape(InputNo,)}")
    print(f"예측된 출력: {predicted_output}")
    print(f"실제 출력: {true_output}")
    print(f"정확 여부: {'맞음' if np.array_equal(predicted_output, true_output) else '틀림'}\n")

accuracy = correct_predictions / total_predictions
print(f"총 정확도: {accuracy * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
입력: [8 4 4 4 4 2 7 8 7 1]
예측된 출력: [1 7 8 7 2 4 4 4 4 8]
실제 출력: [1 7 8 7 2 4 4 4 4 8]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
입력: [8 7 6 1 1 5 6 1 4 3]
예측된 출력: [3 4 1 6 5 1 1 6 7 8]
실제 출력: [3 4 1 6 5 1 1 6 7 8]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
입력: [3 0 9 6 5 2 6 8 9 6]
예측된 출력: [6 9 8 6 2 5 6 9 0 3]
실제 출력: [6 9 8 6 2 5 6 9 0 3]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
입력: [2 7 2 6 4 7 7 0 3 5]
예측된 출력: [5 3 0 7 7 4 6 2 7 2]
실제 출력: [5 3 0 7 7 4 6 2 7 2]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
입력: [7 8 2 1 7 2 8 4 7 1]
예측된 출력: [1 7 4 8 2 7 1 2 8 7]
실제 출력: [1 7 4 8 2 7 1 2 8 7]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
입력: [0 0 0 7 9 7 0 2 2 2]
예측된 출력: [2 2 2 0 7 9 7 0 0 0]
실제 출력: [2 2 2 0 7 9 7 0 0 0]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
입력: [7 3 9 4 8 4 4 0 8 2]
예측된 출력: [2 8 0 4 4 8 4 9 3 7]
실제 출력: [2 8 0 4 4 8 4 9 3 7]
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
입력: [2 7 4 9 3 8 2 9 1 6